# Tarea 5.9
**Benjamín Collao**

## Demostración

El `lufact` de la clase parte con $L = I$ y $U = 0$, y en cada paso $k = 0, \dots, n-2$ hace:

- `U[k,:] = Ak[k,:]`
- `L[:,k] = Ak[:,k] / U[k,k]`
- $A_k \leftarrow A_k - \mathbf{l}_k \mathbf{u}_k^T$ (producto externo)

Queremos mostrar que con eso $L$ queda triangular inferior con unos en la diagonal, $U$ queda triangular superior, y por lo tanto el determinante de $A$ se reduce al producto de la diagonal de $U$.

### 1. Forma de $L$ y $U$

Veamos qué pasa con las entradas de la matriz de trabajo después del paso $k$. El producto externo $\mathbf{l}_k \mathbf{u}_k^T$ tiene como entradas $(\mathbf{l}_k \mathbf{u}_k^T)_{ij} = L_{ik} U_{kj}$. Por la forma en que se calculan $L_{ik}$ y $U_{kj}$:

- Para $i > k$: $L_{ik} = A_{ik}^{(k)} / U_{kk}$, que es justo lo que hace que la resta deje $A_{ik}^{(k+1)} = 0$ para $i > k$.
- Para $j > k$: $U_{kj} = A_{kj}^{(k)}$, que es lo que se copia a $U$.

Entonces, después del paso $k$, la columna $k$ por debajo de la diagonal queda en cero, y la fila $k$ desde la diagonal hacia la derecha ya quedó guardada en $U$. Como esto vale para cada $k$, tenemos:

- $U_{ij} = 0$ si $i > j$. En efecto, la fila $i$ de $U$ se llena recién en el paso $i$ (o en el último paso para $i = n-1$), y en ese momento las columnas $j < i$ ya habían sido anuladas en pasos anteriores. Por lo tanto $U$ es triangular superior.
- $L_{ij} = 0$ si $i < j$. La columna $j$ de $L$ se asigna en el paso $j$ como `Ak[:,j] / pivot`, y para $i < j$ esa entrada de $A_k$ vale cero, porque la fila $i$ de $A_k$ ya quedó completamente anulada en el paso $i$ (se le resta $L_{ii}\mathbf{u}_i^T = \mathbf{u}_i^T$, que es justo esa fila). Por lo tanto $L$ es triangular inferior.
- $L_{kk} = 1$ para todo $k$. Para $k = 0, \dots, n-2$, $L_{kk} = U_{kk}/U_{kk} = 1$. Para $k = n-1$, $L_{n-1,n-1}$ nunca se toca porque el loop va hasta $n-2$, y como partimos con `np.eye`, ya vale 1.

### 2. Determinante de una matriz triangular

Sea $T$ triangular (superior o inferior) de $n \times n$. Afirmo que $\det(T) = \prod_{i=1}^n T_{ii}$. Lo pruebo por inducción en $n$.

**Caso base $n = 1$:** trivial, $\det(T) = T_{11}$.

**Paso inductivo:** supongamos que vale para matrices triangulares de $(n-1) \times (n-1)$.

Si $T$ es triangular superior, expandimos por cofactores a lo largo de la primera columna. Los únicos términos no nulos son los que multiplican a $T_{11}$, porque $T_{i1} = 0$ para $i > 1$. Entonces

$$\det(T) = T_{11} \cdot (-1)^{1+1} \det(T'),$$

donde $T'$ es la submatriz que queda al sacar la primera fila y la primera columna. Veamos que $T'$ también es triangular superior: sus entradas son $T'_{ij} = T_{i+1,j+1}$, y como $T$ es triangular superior, $T_{i+1,j+1} = 0$ cuando $i+1 < j+1$, es decir cuando $i < j$. Por lo tanto $T'$ es triangular superior, y por hipótesis inductiva $\det(T') = \prod_{i=2}^n T_{ii}$. Entonces $\det(T) = T_{11} \prod_{i=2}^n T_{ii} = \prod_{i=1}^n T_{ii}$.

Si $T$ es triangular inferior, el argumento es análogo pero expandiendo por la primera fila. La única entrada no nula de esa fila es $T_{11}$, así que

$$\det(T) = T_{11} \cdot (-1)^{1+1} \det(T'),$$

y $T'$ es triangular inferior porque $T'_{ij} = T_{i+1,j+1} = 0$ cuando $i+1 > j+1$, o sea $i > j$. Por inducción $\det(T') = \prod_{i=2}^n T_{ii}$, y queda $\det(T) = \prod_{i=1}^n T_{ii}$.

### 3. Cierre

Como $L$ es triangular inferior con $L_{ii} = 1$, por el punto anterior $\det(L) = \prod_i L_{ii} = 1$. Como $U$ es triangular superior, $\det(U) = \prod_i U_{ii}$. Usando la propiedad multiplicativa del determinante:

$$\det(A) = \det(LU) = \det(L)\det(U) = 1 \cdot \prod_i U_{ii}.$$

Por lo tanto $\det(A) = \det(U) = \prod_i U_{ii}$. $\blacksquare$

## Código

Primero el `lufact` tal cual lo vimos en clase, y después `determinante`, que factoriza y multiplica la diagonal de $U$.

In [1]:
import numpy as np

def lufact(A):
    n = A.shape[0]
    L = np.eye(n, dtype=float)          # unos en la diagonal
    U = np.zeros((n, n), dtype=float)
    Ak = A.copy().astype(float)

    # reducción por productos externos (lo de la clase)
    for k in range(n - 1):
        U[k, :] = Ak[k, :]
        pivot = U[k, k]
        if pivot == 0:
            raise ZeroDivisionError(f"pivote cero en el índice {k}")
        L[:, k] = Ak[:, k] / pivot
        Ak -= np.outer(L[:, k], U[k, :])

    U[n - 1, n - 1] = Ak[n - 1, n - 1]
    return L, U


def determinante(A):
    # det(A) = det(L) det(U) = 1 * prod(U_ii)
    L, U = lufact(np.asarray(A, dtype=float))
    return np.prod(np.diag(U))

## Verificación con tres matrices

Uso la de 3×3 de la clase, una de 4×4 y una de 5×5. Comparo con `np.linalg.det` y reviso que $LU$ reproduzca $A$.

In [2]:
A1 = np.array([[2, 1, 1],
               [4, -6, 0],
               [-2, 7, 2]])

A2 = np.array([[4, 3, 2, 1],
               [3, 5, 1, 2],
               [2, 1, 6, 3],
               [1, 2, 3, 7]])

A3 = np.array([[1, 2, 0, 1, 3],
               [2, 5, 1, 0, 1],
               [0, 1, 4, 2, 1],
               [1, 0, 2, 6, 2],
               [3, 1, 1, 2, 7]])

print("matriz  n  det(LU)  det(numpy)  dif absoluta  A=LU?")
for nombre, A in [("A1", A1), ("A2", A2), ("A3", A3)]:
    L, U = lufact(A)
    d_lu = determinante(A)
    d_np = np.linalg.det(A)
    ok = np.allclose(L @ U, A)
    print(f"{nombre}      {A.shape[0]}  {d_lu:.6f}  {d_np:.6f}  {abs(d_lu - d_np):.2e}  {ok}")

matriz  n  det(LU)  det(numpy)  dif absoluta  A=LU?
A1      3  -16.000000  -16.000000  1.78e-15  True
A2      4  252.000000  252.000000  5.68e-14  True
A3      5  -576.000000  -576.000000  1.14e-13  True


En $A_1$ se puede ver a mano: la diagonal de $U$ es $(2, -8, 1)$, y $2 \cdot (-8) \cdot 1 = -16$. Miremos esa factorización:

In [3]:
L, U = lufact(A1)
print("L:\n", L)
print("U:\n", U)
print("producto de la diagonal de U:", np.prod(np.diag(U)))

L:
 [[ 1. -0.  0.]
 [ 2.  1.  0.]
 [-1. -1.  1.]]
U:
 [[ 2.  1.  1.]
 [ 0. -8. -2.]
 [ 0.  0.  1.]]
producto de la diagonal de U: -16.0
